# 06 — Blocked nested model selection

**Objective.** Run expanding-window model/configuration selection on 2005–2007 only, estimate pooled out-of-time performance, and construct empirical near-optimal sets.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Permitted block: 2005–2007 only. No 2008–2010 case is used for model-family or hyperparameter selection.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("06", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import pandas as pd
from cruxvc.io import read_table, write_table
from cruxvc.manifest import signing_key_from_environment, verify_protocol_lock
from cruxvc.models import blocked_model_selection, load_model_grid, select_near_optimal_models, startup_bootstrap_metric_se

verify_protocol_lock(P.locks / "phase0_lock.json", signing_key=signing_key_from_environment(), require_hmac=False)
features = read_table(P.processed / "features_strict.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
folds = read_table(P.protocol / "development_folds.parquet")
seed_table = pd.read_csv(P.protocol / "seed_registry.csv")
CTX.recorder.inputs.extend([
    P.locks / "phase0_lock.json", P.processed / "features_strict.parquet", P.processed / "cohort_labels.parquet",
    P.protocol / "development_folds.parquet", P.config / "model_grids.yaml"
])

In [ ]:
grid = load_model_grid(P.config / "model_grids.yaml", int(PROFILE["max_configs_per_family"]))
seeds = seed_table[seed_table["purpose"].eq("model_selection")].sort_values("index")["seed"].head(int(PROFILE["selection_seeds"])).astype(int).tolist()
identifiers = {"case_id", "company_permalink", "t0"}
feature_columns = [column for column in features.columns if column not in identifiers]
result = blocked_model_selection(
    features,
    cohort,
    folds,
    grid,
    outcomes=CFG["outcomes"]["confirmatory"],
    seeds=seeds,
    feature_columns=feature_columns,
    model_dir=P.models / "development_selection",
)

In [ ]:
oof_path = write_table(result.predictions, P.predictions / "development_oof_predictions.parquet")
fold_metrics_path = write_table(result.metrics, P.results / "inference" / "development_fold_metrics.csv")
registry_path = write_table(result.registry, P.models / "development_model_registry.parquet")
se_repetitions = 250 if PROFILE["name"] == "smoke" else 1000
metric_se = startup_bootstrap_metric_se(
    result.predictions,
    metric="log_loss",
    repetitions=se_repetitions,
    seed=int(CFG["execution"]["random_seed"]) + 60,
)
se_path = write_table(metric_se, P.results / "inference" / "development_logloss_bootstrap_se.csv")
near_optimal = select_near_optimal_models(
    result.registry,
    metric_se,
    cap_per_family=int(CFG["explanations"]["maximum_models_per_family"]),
    ap_lift_min=float(CFG["endpoint_gates"]["development_ap_lift_min"]),
    require_positive_brier_skill=bool(CFG["endpoint_gates"]["require_positive_brier_skill"]),
)
near_path = write_table(near_optimal, P.models / "near_optimal_registry.parquet")

In [ ]:
family_gate = (
    near_optimal[near_optimal["near_optimal_global"]]
    .groupby("outcome")["family"].nunique()
    .rename("global_near_optimal_family_count")
    .reset_index()
)
family_gate["passes_two_family_gate"] = family_gate["global_near_optimal_family_count"].ge(2)
family_gate_path = write_table(family_gate, P.audits / "06_near_optimal_family_gate.csv")
if not family_gate["passes_two_family_gate"].all():
    print("WARNING: at least one outcome lacks two global near-optimal families; explanation claims will be gated.")

In [ ]:
CTX.recorder.complete([oof_path, fold_metrics_path, registry_path, se_path, near_path, family_gate_path], extra={"profile": PROFILE["name"], "selection_seeds": seeds})
print(near_optimal.groupby(["outcome", "family"])[["near_optimal_global", "near_optimal_family"]].sum())